# 2. Open RadDB object and filter

Tutorial 1 wrote an archive. This one reads it back and filters it down.

**`RadDB` is one class with two roles.**

| role | what it is |
|---|---|
| *archive-bound* | knows where an archive lives, and reads from it |
| *data-carrying*  | holds the gates you loaded, and narrows them down |

`open()` turns the first into the second. Every operation on a data-carrying
RadDB returns a **new** one, so calls chain and nothing is changed in place.

---

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import polars as pl

import raddb

In [ ]:
# --------------------------------------------------------------------------
# CONFIGURATION — edit these three paths to point at your own data
# --------------------------------------------------------------------------
# ARCHIVE_DIR must be the same archive tutorial 1 wrote.

MCH_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/MCH_datatree_zarr").expanduser()
NEXRAD_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

print("MCH DataTrees   :", MCH_DIR)
print("NEXRAD DataTrees:", NEXRAD_DIR)
print("Archive         :", ARCHIVE_DIR)

In [ ]:
# This notebook stands on its own: build the archive if tutorial 1 has not run.
if not (ARCHIVE_DIR / "L" / "LUT").exists():
    print("building the archive (see tutorial 1) ...")
    raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=2056).archive(datatree_dir=MCH_DIR)
else:
    print("archive already present:", ARCHIVE_DIR)

## 1. `open()`: reading the archive

Reading never needs a CRS: it is recovered from the archive itself.

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
rdf = db.open(radars="L")
rdf.head()

`open()` narrows *before* anything is loaded — the time range, the radars and the
columns are all pushed down into the Parquet scan, so you never pay for data you
did not ask for.

In [ ]:
# Only two variables, only radar L
small_df = db.open(radars="L", columns=["DBZH", "ZDR"])
print(f"small_df:\tcolumns: {small_df.columns()}\nsmall_df:\tgates: {len(small_df)}")
print("-------------------------------")
# time period
day_df = db.open(radars="L", time_period=("2024-06-12", "2024-06-13"))
print(f"day_df:\t\tcolumns: {day_df.columns()}\nday_df:\t\tgates: {len(day_df):,}")

In [ ]:
# Filters can be pushed down at open() too, so filtered-out rows are never materialised
filtered_df = db.open(radars="L", filters={"var": "DBZH", "logic": ">", "threshold": 30})
print(f"before:\t{len(rdf):,} gates\t(with DBZH > 0 dBz)\nafter:\t{len(filtered_df):,}  gates\t(with DBZH > 30 dBz)")

## 2. What you are holding

The data lives in `.data` as a **polars** DataFrame. Polars is the backend
throughout RadDB (the read path, the LUT, the archive writer).

In [ ]:
print("type:\t\t", type(rdf.data))
print("name type:\t", type(rdf.data).__name__)
print("shape:\t\t", rdf.data.shape)
rdf.data.head()

In [ ]:
print("radars    :", rdf.radars())
print("variables :", rdf.columns())
print("time range:", rdf.start_time(), "->", rdf.end_time())
print("lon/lat    :", [round(v, 3) for v in rdf.geographic_extent()])
print("archive CRS:", rdf.crs())  # recovered from the archive itself

## 3. `filter()`: threshold on values

A filter is a plain dict: `{"var", "logic", "threshold"}`

In [ ]:
rain = rdf.filter({"var": "DBZH", "logic": ">", "threshold": 20})
print(f"DBZH > 20: {len(rain):,} gates")

filt_df = rdf.filter(
    [
        {"var": "DBZH", "logic": ">", "threshold": 20},
        {"var": "RHOHV", "logic": ">=", "threshold": 0.98},
        {"var": "ZDR", "logic": ">", "threshold": 4},
    ],
)
print(f"DBZH > 20, RHOHV >= 0.98, ZDR > 4 : {len(filt_df):,} gates")

## 4. `sel()`: select by label, xarray-style

Where `filter()` thresholds *values*, `sel()` selects by **coordinate**: a time, a
sweep, a range window, a longitude/latitude box. Scalars match exactly, `slice`
gives a closed interval, and a list matches any of its members.

In [ ]:
print("one sweep      :", f"{len(rdf.sel(sweep=1)):,}")
print("sweeps 1,2,3   :", f"{len(rdf.sel(sweep=[1, 2, 3])):,}")
print("range 10-50 km :", f"{len(rdf.sel(range=slice(10_000, 50_000))):,}")
print("a lon/lat box  :", f"{len(rdf.sel(lon=slice(8.6, 9.0), lat=slice(46.0, 46.4))):,}")

`range`, `azimuth`, `elevation_angle`, `latitude`, `longitude`
and `altitude` are **not stored in the Parquet files** — they live once in the LUT.
`sel()` borrows the column it needs, evaluates the selection, and drops it again,
so selecting on geometry costs no storage.

In [ ]:
print("stored per gate:", rdf.columns())
print("also selectable :", ["range", "azimuth", "elevation_angle", "latitude", "longitude", "altitude", "sweep"])

narrow = rdf.sel(sweep=1, range=slice(20_000, 60_000))
print(f"\nsweep 1, 20-60 km: {len(narrow):,} gates " f"(columns unchanged: {narrow.columns() == rdf.columns()})")

## 5. `add_feature()`: compute columns

`add_feature()` adds a column derived from the ones you already have and returns a
new RadDB, so it drops straight into a pipeline. The function receives the polars
frame; return a Series, a numpy array, or a polars expression.

In [ ]:
derived = rdf.add_feature("DBZH_lin", lambda df: 10 ** (df["DBZH"] / 10)).add_feature(
    "DBZH_dev",
    lambda df: df["DBZH"] - df["DBZH"].mean(),
)
derived.head()

If you would rather work in plain polars or pandas, nothing stops you — `.data`
is an ordinary polars frame, and `to_pandas()` gives an ordinary pandas one.

In [ ]:
rdf.data.with_columns((pl.col("DBZH") - pl.col("ZDR")).alias("DIFF"))

df = rdf.to_pandas()
df["DIFF"] = df["DBZH"] - df["ZDR"]

print(f"rdf type: {type(rdf.data)}")
print(f"df  type: {type(df)}")
df.head()

## 6. Framework converter

The gates can leave RadDB as a pandas DataFrame, a geopandas GeoDataFrame, or an
xarray DataTree — three converters for three different frameworks.

### `to_pandas()`: the DataFrame

`to_pandas()` returns the loaded gates as an ordinary pandas DataFrame. On its own
it hands back exactly what is stored per gate: `gate_id`, `time`, the polarimetric variables, and
the `volume_time` / `radar` labels.

Geometry is **not** stored per gate — it lives once in the LUT — so it is merged
on `gate_id` only when you ask for it:

| call | columns added |
|---|---|
| `to_pandas()` | nothing; the stored columns only (dynamic variables) |
| `to_pandas(with_geometry=True)` | `latitude`, `longitude`, `altitude`, `sweep` |
| `to_pandas(with_polar_coords=True)` | the same, **plus** `range`, `azimuth`, `elevation_angle` |

`with_polar_coords` implies `with_geometry`. The polar coordinates are off by
default because they repeat what the Cartesian columns already say, unless you are
working in polar space.

Note what is **not** included: `x`, `y`, `z` — metres from the radar — are never
added by either flag, and the projected `x_<epsg>` / `y_<epsg>` appear only under a
condition. The next three cells explain why, and how to load all of them.

In [ ]:
# No flags: the stored columns only, exactly as open() loaded them.
df = rain.to_pandas()
print("to_pandas():", type(df).__name__, df.shape)
print(list(df.columns))

In [ ]:
# with_geometry=True joins the per-gate coordinates from the LUT on gate_id.
df_geo = rain.to_pandas(with_geometry=True)
print("added by with_geometry     :", [c for c in df_geo.columns if c not in df.columns])

In [ ]:
# with_polar_coords=True also brings the polar coordinates the geometry came from.
df_polar = rain.to_pandas(with_polar_coords=True)
print("added by with_polar_coords :", [c for c in df_polar.columns if c not in df.columns])
df_polar.head(3)

### Where the geometry lives

Two things are easy to trip over:

- **`x_<epsg>` / `y_<epsg>` appear only if the RadDB was created with `crs=`.**
  `crs()` reports the archive's projection either way, but the converters add the
  projected pair only when a projection was asked for explicitly.
- **`x` / `y` / `z`** — metres from the radar — are LUT columns that no converter
  attaches. Join the LUT yourself to get them, or any other LUT column.

In [ ]:
# Projected coordinates: state the CRS when creating the RadDB, and
# with_geometry=True then adds x_<epsg> / y_<epsg> alongside lon/lat/alt.
db_proj = raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=2056)
rain_proj = db_proj.open(radars="L", filters={"var": "DBZH", "logic": ">", "threshold": 20})

print("without crs= :", list(rain.to_pandas(with_geometry=True).columns))
print("with crs=2056:", list(rain_proj.to_pandas(with_geometry=True).columns))

In [ ]:
# Any LUT column can be attached by joining on gate_id.  This is also how you add
# geometry to a frame loaded with open(), and the only way to get x / y / z
# (metres from the radar), which no converter attaches.
geometry = db.get_lut("L").select(["gate_id", "x", "y", "z", "x_2056", "y_2056"])
joined = rain.data.join(geometry, on="gate_id", how="left")
joined.select(["gate_id", "DBZH", "x", "y", "z", "x_2056", "y_2056"]).head()

### `to_geopandas()` — points with a CRS

In [ ]:
# geopandas: point geometry per gate, ready for spatial joins or QGIS
gdf = rain.to_geopandas()
print("to_geopandas: ", type(gdf))
print("CRS:", gdf.crs)
gdf[["gate_id", "DBZH", "geometry"]].head()

### `to_datatree()` — back to xarray

In [ ]:
# DataTree: the full polar structure, for xarray workflows.
# A DataTree describes ONE volume: each sweep is an (azimuth x range) grid and
# time is a per-ray coordinate, so there is no dimension to stack volumes along.
# Choose which volume to rebuild; to_datatree() then NaN-fills the gates that
# were filtered out, restoring the complete azimuth x range grid.
volumes = rdf.data["volume_time"].unique().sort().to_list()
print(f"{len(volumes)} volumes loaded -> rebuilding the first one\n")

dt = rdf.to_datatree(timestep=volumes[0])
dt

---
**Next:** [3 — Areas of interest](03_area_of_interest.ipynb)